In [1]:
"""
build_master_corpus.py
======================
Deprem projesi: 8 kaynak CSV'sini ortak şemada tek bir corpus'a birleştirir.

Birleşik şema (master_corpus.csv kolonları):
    doc_id   — kaynak ön-ekli benzersiz ID, örn. "tccb_0001"
    source   — kısa kaynak kodu, örn. "TCCB"
    date     — ISO YYYY-MM-DD (parse edilemediyse boş)
    url      — kaynak URL
    text     — temizlenmiş tam metin

Kullanım:
    cd /Users/mehmetbagdinli/Desktop/deprem
    python build_master_corpus.py
"""

import re
import unicodedata
from pathlib import Path

import pandas as pd

# ─── Ayarlar ─────────────────────────────────────────────────────────────────
BASE_DIR = Path("/Users/mehmetbagdinli/Desktop/deprem")
OUT_CSV  = BASE_DIR / "master_corpus.csv"
META_CSV = BASE_DIR / "master_corpus_meta.csv"

# 8 kaynak — her biri ortak şemaya nasıl çevrileceğini söylüyor
# id_prefix → doc_id'lerin başında kullanılır (örn. "tccb_0001")
SOURCES = [
    {
        "file":        "tccb_konusmalar_p1_p15.csv",
        "id_prefix":   "tccb",
        "source":      "TCCB",
        "source_full": "Cumhurbaşkanlığı Konuşmaları",
        "source_type": "cumhurbaskanligi",
        "text_cols":   ["content"],
        "url_col":     "link",
        "title_col":   "title",
        "date_col":    "date",
    },
    {
        "file":        "afad_haberler.csv",
        "id_prefix":   "afad",
        "source":      "AFAD",
        "source_full": "Afet ve Acil Durum Yönetimi Başkanlığı",
        "source_type": "merkezi_kurum",
        "text_cols":   ["full_text", "teaser"],
        "url_col":     "url",
        "title_col":   "title",
        "date_col":    "date",
    },
    {
        "file":        "csb_haberler.csv",
        "id_prefix":   "csb",
        "source":      "CSB",
        "source_full": "Çevre, Şehircilik ve İklim Değişikliği Bakanlığı",
        "source_type": "bakanlik",
        "text_cols":   ["full_text", "summary"],
        "url_col":     "url",
        "title_col":   "title",
        "date_col":    "date",
    },
    {
        "file":        "hatay_haberler.csv",
        "id_prefix":   "hatay_val",
        "source":      "Hatay Valiliği",
        "source_full": "Hatay Valiliği",
        "source_type": "valilik",
        "text_cols":   ["full_text", "teaser"],
        "url_col":     "url",
        "title_col":   "title",
        "date_col":    "date",
    },
    {
        "file":        "hatay_belediye_haberler.csv",
        "id_prefix":   "hatay_bld",
        "source":      "Hatay Belediye",
        "source_full": "Hatay Büyükşehir Belediyesi",
        "source_type": "belediye",
        "text_cols":   ["full_text"],
        "url_col":     "url",
        "title_col":   "title",
        "date_col":    "date",
    },
    {
        "file":        "kahramanmaras_haberler.csv",
        "id_prefix":   "kmaras_val",
        "source":      "Kahramanmaraş Valiliği",
        "source_full": "Kahramanmaraş Valiliği",
        "source_type": "valilik",
        "text_cols":   ["full_text", "teaser"],
        "url_col":     "url",
        "title_col":   "title",
        "date_col":    "date",
    },
    {
        "file":        "kmaras_belediye_haberler.csv",
        "id_prefix":   "kmaras_bld",
        "source":      "Kahramanmaraş Belediye",
        "source_full": "Kahramanmaraş Büyükşehir Belediyesi",
        "source_type": "belediye",
        "text_cols":   ["full_text", "teaser"],
        "url_col":     "url",
        "title_col":   "title",
        "date_col":    "date",
    },
    {
        "file":        "adiyaman_haberler.csv",
        "id_prefix":   "adiyaman_val",
        "source":      "Adıyaman Valiliği",
        "source_full": "Adıyaman Valiliği",
        "source_type": "valilik",
        "text_cols":   ["full_text", "teaser"],
        "url_col":     "url",
        "title_col":   "title",
        "date_col":    "date",
    },
    # NOT: adiyaman_vali_haberler.csv hâlâ Apple Numbers formatında.
    # Numbers'da File → Export To → CSV ile çevirip aynı klasöre koyduktan sonra
    # bu listeye id_prefix="adiyaman_vali" ile bir blok daha ekleyebilirsin.
]

# ─── Tarih ayrıştırma ────────────────────────────────────────────────────────
TURK_AYLAR = {
    "ocak": 1, "şubat": 2, "subat": 2, "mart": 3, "nisan": 4,
    "mayıs": 5, "mayis": 5, "haziran": 6, "temmuz": 7, "ağustos": 8,
    "agustos": 8, "eylül": 9, "eylul": 9, "ekim": 10, "kasım": 11,
    "kasim": 11, "aralık": 12, "aralik": 12,
}

RE_ISO     = re.compile(r"\b(\d{4})-(\d{2})-(\d{2})\b")
RE_NUMERIC = re.compile(r"\b(\d{1,2})[.\-/](\d{1,2})[.\-/](\d{4})\b")
RE_TURKISH = re.compile(
    r"\b(\d{1,2})\s+"
    r"(Ocak|Şubat|Subat|Mart|Nisan|Mayıs|Mayis|Haziran|"
    r"Temmuz|Ağustos|Agustos|Eylül|Eylul|Ekim|Kasım|Kasim|Aralık|Aralik)\s+"
    r"(\d{4})\b",
    re.IGNORECASE,
)


def parse_date(value):
    """Çeşitli tarih formatlarını ISO (YYYY-MM-DD) string'ine çevirir.
    Olmazsa (ham_metin, None) döndürür."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "", None
    raw = str(value).strip()
    if not raw or raw.lower() in {"nan", "none", "nat"}:
        return "", None

    m = RE_ISO.search(raw)
    if m:
        y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
        if 1 <= d <= 31 and 1 <= mo <= 12:
            return raw, f"{y:04d}-{mo:02d}-{d:02d}"

    m = RE_NUMERIC.search(raw)
    if m:
        d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        if 1 <= d <= 31 and 1 <= mo <= 12:
            return raw, f"{y:04d}-{mo:02d}-{d:02d}"

    m = RE_TURKISH.search(raw)
    if m:
        d  = int(m.group(1))
        mo = TURK_AYLAR[m.group(2).lower()]
        y  = int(m.group(3))
        if 1 <= d <= 31 and 1 <= mo <= 12:
            return raw, f"{y:04d}-{mo:02d}-{d:02d}"

    return raw, None


# ─── Metin temizleme ─────────────────────────────────────────────────────────
def clean_text(value):
    """Whitespace + Unicode normalizasyonu."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    s = str(value)
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\r\n", "\n").replace("\r", "\n").replace("\xa0", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def first_nonempty_text(row, cols):
    """Verilen kolonlardan ilk dolu olanı döndürür."""
    for c in cols:
        if c not in row:
            continue
        cleaned = clean_text(row[c])
        if cleaned:
            return cleaned
    return ""


# ─── Ana akış ────────────────────────────────────────────────────────────────
def main():
    if not BASE_DIR.exists():
        raise FileNotFoundError(f"{BASE_DIR} bulunamadı.")

    all_rows = []
    summary  = []

    print(f"\nBASE_DIR: {BASE_DIR}\n")

    for spec in SOURCES:
        path = BASE_DIR / spec["file"]
        if not path.exists():
            print(f"  ⚠ {spec['file']}  → bulunamadı, atlandı.")
            continue

        try:
            df = pd.read_csv(path)
        except Exception as e:
            print(f"  ✗ {spec['file']}  → okunamadı: {e}")
            continue

        # Bu kaynaktan satırları topla
        rows_for_source = []
        for _, r in df.iterrows():
            text     = first_nonempty_text(r, spec["text_cols"])
            url      = clean_text(r.get(spec["url_col"], ""))
            title    = clean_text(r.get(spec["title_col"], ""))
            d_raw, d_iso = parse_date(r.get(spec["date_col"]))

            rows_for_source.append({
                "source": spec["source"],
                "date":   d_iso or "",
                "url":    url,
                "text":   text,
            })

        # Tarihe göre sırala (boş tarihliler sona)
        sub = pd.DataFrame(rows_for_source)
        sub["_sort_key"] = sub["date"].replace("", "9999-99-99")
        sub = sub.sort_values("_sort_key").drop(columns="_sort_key").reset_index(drop=True)

        # Kaynak içi ID ata: tccb_0001, afad_001 vs.
        # Genişlik satır sayısına göre (en az 3 hane)
        width = max(3, len(str(len(sub))))
        sub.insert(0, "doc_id", [
            f"{spec['id_prefix']}_{i+1:0{width}d}" for i in range(len(sub))
        ])

        all_rows.append(sub)

        n_text = (sub["text"].str.len() > 0).sum()
        n_date = (sub["date"] != "").sum()

        summary.append({
            "source":       spec["source"],
            "source_full":  spec["source_full"],
            "source_file":  spec["file"],
            "n_rows":       len(sub),
            "n_with_text":  int(n_text),
            "n_with_date":  int(n_date),
            "pct_text":     round(100 * n_text / max(len(sub), 1), 1),
            "pct_date":     round(100 * n_date / max(len(sub), 1), 1),
            "id_first":     sub["doc_id"].iloc[0]  if len(sub) else "",
            "id_last":      sub["doc_id"].iloc[-1] if len(sub) else "",
        })

        print(f"  ✓ {spec['file']:36s}  {len(sub):>5} satır  "
              f"({sub['doc_id'].iloc[0]} → {sub['doc_id'].iloc[-1]})")

    # ─── Birleştir ───────────────────────────────────────────────────────────
    corpus = pd.concat(all_rows, ignore_index=True)

    # Aynı kaynaktan aynı URL → tek tut
    before = len(corpus)
    corpus = corpus.drop_duplicates(subset=["source", "url"], keep="first").reset_index(drop=True)
    dedup_dropped = before - len(corpus)

    # Final kolon sırası — sadece istenen 5 kolon
    corpus = corpus[["doc_id", "source", "date", "url", "text"]]

    # ─── Özet ────────────────────────────────────────────────────────────────
    summary_df = pd.DataFrame(summary)

    print(f"\n{'─'*70}")
    print(f"  Birleşik Corpus Özeti")
    print(f"{'─'*70}")
    print(f"   Toplam satır            : {len(corpus)}")
    print(f"   Metin dolu              : {(corpus['text'].str.len() > 0).sum()}")
    print(f"   Tarihi dolu             : {(corpus['date'] != '').sum()}")
    print(f"   Mükerrer (silindi)      : {dedup_dropped}")
    print(f"   Kaynak sayısı           : {corpus['source'].nunique()}")
    dates = corpus.loc[corpus["date"] != "", "date"]
    if len(dates):
        print(f"   Tarih aralığı           : {dates.min()} → {dates.max()}")

    # ─── Yaz ─────────────────────────────────────────────────────────────────
    corpus.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    summary_df.to_csv(META_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✓ Yazıldı: {OUT_CSV}")
    print(f"✓ Yazıldı: {META_CSV}")


if __name__ == "__main__":
    main()


BASE_DIR: /Users/mehmetbagdinli/Desktop/deprem

  ✓ tccb_konusmalar_p1_p15.csv              600 satır  (tccb_001 → tccb_600)
  ✓ afad_haberler.csv                        70 satır  (afad_001 → afad_070)
  ✓ csb_haberler.csv                       1500 satır  (csb_0001 → csb_1500)
  ✓ hatay_haberler.csv                      460 satır  (hatay_val_001 → hatay_val_460)
  ✓ hatay_belediye_haberler.csv            2408 satır  (hatay_bld_0001 → hatay_bld_2408)
  ✓ kahramanmaras_haberler.csv              710 satır  (kmaras_val_001 → kmaras_val_710)
  ✓ kmaras_belediye_haberler.csv           4410 satır  (kmaras_bld_0001 → kmaras_bld_4410)
  ✓ adiyaman_haberler.csv                    70 satır  (adiyaman_val_001 → adiyaman_val_070)

──────────────────────────────────────────────────────────────────────
  Birleşik Corpus Özeti
──────────────────────────────────────────────────────────────────────
   Toplam satır            : 10218
   Metin dolu              : 10060
   Tarihi dolu             : 10215

In [2]:
import pandas as pd

BASE = "/Users/mehmetbagdinli/Desktop/deprem"

df = pd.read_csv(f"{BASE}/master_corpus.csv", encoding="utf-8-sig",
                 engine="python", on_bad_lines="skip")
print(f"Ham kayıt: {len(df)}")

# ── 1. Exact duplicate temizle ────────────────────────────────────────────────
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Duplicate sonrası: {len(df)}")

# ── 2. Çok kısa metinleri at ──────────────────────────────────────────────────
df["kelime"] = df["text"].astype(str).str.split().str.len()
df = df[df["kelime"] >= 50].reset_index(drop=True)
print(f"Kısa metin sonrası: {len(df)}")

# ── 3. Tarih filtresi: Şubat 2023 – Aralık 2024 ──────────────────────────────
df["date"] = pd.to_datetime(df["date"], errors="coerce")
onceki = len(df)
df = df[
    (df["date"] >= "2023-02-01") &
    (df["date"] <= "2024-12-31")
].reset_index(drop=True)
print(f"Tarih filtresi sonrası: {len(df)}  ({onceki - len(df)} metin aralık dışı)")

# ── 4. Deprem anahtar kelime filtresi ─────────────────────────────────────────
ANAHTAR = [
    "deprem", "enkaz", "afet", "konut", "yıkım", "yıkılan", "hasar",
    "kurtarma", "tahliye", "konteyner", "prefabrik", "kalıcı konut",
    "altyapı", "yeniden yapım", "yeniden yapılanma", "dönüşüm",
    "inşaat", "TOKİ", "AFAD", "arama kurtarma", "depremzede",
    "6 şubat", "kahramanmaraş depremi", "hatay depremi",
    "acil durum", "barınak", "çadır kent"
]
pattern = "|".join(ANAHTAR)
df["depremle_ilgili"] = df["text"].str.contains(pattern, case=False, na=False)
print(f"Anahtar kelime eşleşen : {df['depremle_ilgili'].sum()}")
print(f"Eşleşmeyen (atılacak)  : {(~df['depremle_ilgili']).sum()}")
df = df[df["depremle_ilgili"]].copy()

# ── 5. Kaynak başına 500 üst sınır (tarih sırasıyla) ─────────────────────────
df = df.sort_values("date")
df = (
    df.groupby("source", group_keys=False)
    .apply(lambda x: x.head(500))
    .reset_index(drop=True)
)

# ── Özet ─────────────────────────────────────────────────────────────────────
print(f"\n── Filtrelenmiş corpus ──────────────────────────────────")
print(f"Toplam kayıt : {len(df)}")
print(f"\nKaynak dağılımı:")
print(df.groupby("source")["text"].count().rename("n").to_string())
print(f"\nTarih aralığı: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Tarihsiz     : {df['date'].isna().sum()}")

# ── 6. Kaydet ─────────────────────────────────────────────────────────────────
df[["source", "date", "text"]].to_csv(
    f"{BASE}/corpus_filtre.csv", index=False, encoding="utf-8-sig"
)
print(f"\n✅ Kaydedildi → corpus_filtre.csv")


Ham kayıt: 10218
Duplicate sonrası: 10048
Kısa metin sonrası: 10015
Tarih filtresi sonrası: 5099  (4916 metin aralık dışı)
Anahtar kelime eşleşen : 3225
Eşleşmeyen (atılacak)  : 1874

── Filtrelenmiş corpus ──────────────────────────────────
Toplam kayıt : 2122

Kaynak dağılımı:
source
AFAD                       33
Adıyaman Valiliği          30
CSB                       500
Hatay Belediye            500
Hatay Valiliği             31
Kahramanmaraş Belediye    500
Kahramanmaraş Valiliği    212
TCCB                      316

Tarih aralığı: 2023-02-02 → 2024-12-28
Tarihsiz     : 0

✅ Kaydedildi → corpus_filtre.csv


/var/folders/fq/hry5knqn05n0lrcxpj8fdw_r0000gn/T/ipykernel_16739/1664870815.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.head(500))


In [27]:
import pandas as pd

BASE = "/Users/mehmetbagdinli/Desktop/deprem"

df = pd.read_csv(f"{BASE}/master_corpus.csv", encoding="utf-8-sig",
                 engine="python", on_bad_lines="skip")
print(f"Ham kayıt: {len(df)}")


df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Duplicate sonrası: {len(df)}")


df["kelime"] = df["text"].astype(str).str.split().str.len()
df = df[df["kelime"] >= 50].reset_index(drop=True)
print(f"Kısa metin sonrası: {len(df)}")


df["date"] = pd.to_datetime(df["date"], errors="coerce")
onceki = len(df)
df = df[
    (df["date"] >= "2023-02-01") &
    (df["date"] <= "2024-12-31")
].reset_index(drop=True)
print(f"Tarih filtresi sonrası: {len(df)}  ({onceki - len(df)} metin aralık dışı)")

ANAHTAR = [
    "deprem", "enkaz", "afet", "konut", "yıkım", "yıkılan", "hasar",
    "kurtarma", "tahliye", "konteyner", "prefabrik", "kalıcı konut",
    "altyapı", "yeniden yapım", "yeniden yapılanma", "dönüşüm",
    "inşaat", "TOKİ", "AFAD", "arama kurtarma", "depremzede",
    "6 şubat", "kahramanmaraş depremi", "hatay depremi",
    "acil durum", "barınak", "çadır kent"
]
pattern = "|".join(ANAHTAR)
df["depremle_ilgili"] = df["text"].str.contains(pattern, case=False, na=False)
print(f"Anahtar kelime eşleşen : {df['depremle_ilgili'].sum()}")
print(f"Eşleşmeyen (atılacak)  : {(~df['depremle_ilgili']).sum()}")
df = df[df["depremle_ilgili"]].copy()


def assign_period(d):
    if pd.isna(d):
        return None
    if d <= pd.Timestamp("2023-04-30"):
        return "1_post_quake"           # Şubat–Nisan 2023 (acil dönem)
    elif d <= pd.Timestamp("2023-05-31"):
        return "2_pres_campaign"        # Mayıs 2023 (CB seçimi)
    elif d <= pd.Timestamp("2024-01-31"):
        return "3_inter_election"       # Haziran 2023 – Ocak 2024
    elif d <= pd.Timestamp("2024-03-31"):
        return "4_local_campaign"       # Şubat–Mart 2024 (yerel seçim)
    else:
        return "5_post_local"           # Nisan–Aralık 2024

df["period"] = df["date"].apply(assign_period)


PROVINCE = {
    "Hatay Belediye":         "hatay",
    "Hatay Valiliği":         "hatay",
    "Kahramanmaraş Belediye": "kahramanmaras",
    "Kahramanmaraş Valiliği": "kahramanmaras",
    "Adıyaman Valiliği":      "adiyaman",
    "TCCB":                   "national",
    "AFAD":                   "national",
    "CSB":                    "national",
}
df["province"] = df["source"].map(PROVINCE)


SOURCE_TYPE = {
    "Hatay Belediye":         "belediye",
    "Hatay Valiliği":         "valilik",
    "Kahramanmaraş Belediye": "belediye",
    "Kahramanmaraş Valiliği": "valilik",
    "Adıyaman Valiliği":      "valilik",
    "TCCB":                   "cumhurbaskanligi",
    "AFAD":                   "afad",
    "CSB":                    "bakanlik",
}
df["source_type"] = df["source"].map(SOURCE_TYPE)


df = df.sort_values("date")
df = (
    df.groupby("source", group_keys=False)
    .apply(lambda x: x.head(500))
    .reset_index(drop=True)
)


print(f"\n── Filtrelenmiş corpus ──────────────────────────────────")
print(f"Toplam kayıt : {len(df)}")
print(f"\nKaynak dağılımı:")
print(df.groupby("source")["text"].count().rename("n").to_string())
print(f"\nDönem dağılımı:")
print(df.groupby("period")["text"].count().rename("n").to_string())
print(f"\nİl dağılımı:")
print(df.groupby("province")["text"].count().rename("n").to_string())

df[["doc_id", "source", "source_type", "province", "period", "date", "text"]].to_csv(
    f"{BASE}/corpus_filtre.csv", index=False, encoding="utf-8-sig"
)
print(f"\n Kaydedildi → corpus_filtre.csv")

Ham kayıt: 10218
Duplicate sonrası: 10048
Kısa metin sonrası: 10015
Tarih filtresi sonrası: 5099  (4916 metin aralık dışı)
Anahtar kelime eşleşen : 3225
Eşleşmeyen (atılacak)  : 1874

── Filtrelenmiş corpus ──────────────────────────────────
Toplam kayıt : 2122

Kaynak dağılımı:
source
AFAD                       33
Adıyaman Valiliği          30
CSB                       500
Hatay Belediye            500
Hatay Valiliği             31
Kahramanmaraş Belediye    500
Kahramanmaraş Valiliği    212
TCCB                      316

Dönem dağılımı:
period
1_post_quake        474
2_pres_campaign     178
3_inter_election    807
4_local_campaign    138
5_post_local        525

İl dağılımı:
province
adiyaman          30
hatay            531
kahramanmaras    712
national         849

✅ Kaydedildi → corpus_filtre.csv


/var/folders/fq/hry5knqn05n0lrcxpj8fdw_r0000gn/T/ipykernel_16739/1687231164.py:89: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.head(500))


In [28]:
import pandas as pd

BASE = "/Users/mehmetbagdinli/Desktop/deprem"


master = pd.read_csv(f"{BASE}/master_corpus.csv", encoding="utf-8-sig",
                     engine="python", on_bad_lines="skip")
corpus = pd.read_csv(f"{BASE}/corpus_filtre.csv", encoding="utf-8-sig",
                     engine="python", on_bad_lines="skip")
ann    = pd.read_csv(f"{BASE}/final_300.csv", encoding="utf-8-sig",
                     engine="python", on_bad_lines="skip")

print(f"Master  : {len(master)} | Corpus filtre: {len(corpus)} | Annotation: {len(ann)}")


def make_key(s):
    return str(s)[:300].strip().lower()

master["key"] = master["text"].apply(make_key)
corpus["key"] = corpus["text"].apply(make_key)
ann["key"]    = ann["text"].apply(make_key)


ann_map = dict(zip(ann["key"], ann["doc_id"]))   # text → annotation_doc_id
print(f"Annotation map: {len(ann_map)} unique key")


corpus["doc_id"] = corpus["key"].map(ann_map).fillna(corpus["doc_id"])

eslesen = corpus["key"].isin(ann_map).sum()
print(f"\nCorpus filtre'de annotation eşleşmesi: {eslesen}/{len(ann)}")


eksik_keys = set(ann["key"]) - set(corpus["key"])
print(f"Eksik annotation metni: {len(eksik_keys)}")

if eksik_keys:
    
    eksik = master[master["key"].isin(eksik_keys)].copy()
    print(f"Master'dan bulunan eksik: {len(eksik)}")

    
    eksik["doc_id"] = eksik["key"].map(ann_map)

    
    ann_extra = ann[["key","period","province","source_type"]].drop_duplicates("key")
    eksik = eksik.merge(ann_extra, on="key", how="left")

    
    eksik["date"] = pd.to_datetime(eksik["date"], errors="coerce")
    eksik = eksik[["doc_id","source","source_type","province","period","date","text"]]


    corpus_full = pd.concat([corpus.drop(columns=["key"]), eksik], ignore_index=True)
    corpus_full = corpus_full.drop_duplicates(subset=["doc_id"]).reset_index(drop=True)
else:
    corpus_full = corpus.drop(columns=["key"])


ortak_id = set(corpus_full["doc_id"]) & set(ann["doc_id"])
print(f"\n── FINAL ──")
print(f"Toplam corpus: {len(corpus_full)}")
print(f"Annotation doc_id eşleşme: {len(ortak_id)}/{len(ann)}")


corpus_full.to_csv(f"{BASE}/corpus_filtre.csv", index=False, encoding="utf-8-sig")
print(f"\n corpus_filtre.csv güncellendi")

Master  : 10218 | Corpus filtre: 2122 | Annotation: 300
Annotation map: 299 unique key

Corpus filtre'de annotation eşleşmesi: 264/300
Eksik annotation metni: 67
Master'dan bulunan eksik: 67

── FINAL ──
Toplam corpus: 2138
Annotation doc_id eşleşme: 299/300

✅ corpus_filtre.csv güncellendi


In [29]:
import pandas as pd

BASE = "/Users/mehmetbagdinli/Desktop/deprem"

ann = pd.read_csv(f"{BASE}/final_300.csv", encoding="utf-8-sig",
                  engine="python", on_bad_lines="skip")
corpus = pd.read_csv(f"{BASE}/corpus_filtre.csv", encoding="utf-8-sig",
                     engine="python", on_bad_lines="skip")

# Annotation'da hangi doc_id corpus'ta yok?
eksik = set(ann["doc_id"]) - set(corpus["doc_id"])
print(f"Eksik doc_id: {eksik}")

# Annotation'da duplicate text var mı?
dup = ann[ann["text"].duplicated(keep=False)].sort_values("text")
print(f"\nAnnotation'da duplicate text: {len(dup)} satır")
if len(dup) > 0:
    print(dup[["doc_id","source","date"]].to_string())

Eksik doc_id: {'afad_032'}

Annotation'da duplicate text: 0 satır


In [30]:
import pandas as pd

BASE = "/Users/mehmetbagdinli/Desktop/deprem"

ann    = pd.read_csv(f"{BASE}/final_300.csv", encoding="utf-8-sig",
                     engine="python", on_bad_lines="skip")
corpus = pd.read_csv(f"{BASE}/corpus_filtre.csv", encoding="utf-8-sig",
                     engine="python", on_bad_lines="skip")
master = pd.read_csv(f"{BASE}/master_corpus.csv", encoding="utf-8-sig",
                     engine="python", on_bad_lines="skip")


afad_row = ann[ann["doc_id"] == "afad_032"].iloc[0]
print(f"Aranan metin (ilk 200 karakter):")
print(repr(afad_row["text"][:200]))
print(f"\nKaynak: {afad_row['source']} | Tarih: {afad_row['date']}")


master["date"] = pd.to_datetime(master["date"], errors="coerce")
adaylar = master[
    (master["source"] == "AFAD") &
    (master["date"] == pd.to_datetime(afad_row["date"]))
]
print(f"\nMaster'da aynı tarih + AFAD: {len(adaylar)} aday")


hedef_tarih = pd.to_datetime(afad_row["date"])
adaylar2 = master[
    (master["source"] == "AFAD") &
    (master["date"].between(hedef_tarih - pd.Timedelta(days=3),
                             hedef_tarih + pd.Timedelta(days=3)))
]
print(f"Master'da ±3 gün AFAD: {len(adaylar2)} aday")
for _, r in adaylar2.head(5).iterrows():
    print(f"\n  doc_id: {r['doc_id']} | tarih: {r['date'].date()}")
    print(f"  text: {repr(str(r['text'])[:150])}")


yeni_satir = pd.DataFrame([{
    "doc_id":      "afad_032",
    "source":      afad_row["source"],
    "source_type": afad_row["source_type"],
    "province":    afad_row["province"],
    "period":      afad_row["period"],
    "date":        pd.to_datetime(afad_row["date"]),
    "text":        afad_row["text"],
}])

corpus_full = pd.concat([corpus, yeni_satir], ignore_index=True)
corpus_full = corpus_full.drop_duplicates(subset=["doc_id"]).reset_index(drop=True)

corpus_full.to_csv(f"{BASE}/corpus_filtre.csv", index=False, encoding="utf-8-sig")

# Doğrula
ortak = set(corpus_full["doc_id"]) & set(ann["doc_id"])
print(f"\n Toplam corpus: {len(corpus_full)}")
print(f" Annotation eşleşme: {len(ortak)}/{len(ann)}")

Aranan metin (ilk 200 karakter):
'Türkiye’nin afet yönetimi ve uluslararası yardım kurumu: AFAD Afet ve acil durumlara ilişkin ülkenin yetkin ve yetkili teşkilatıdır.\n\nAFAD, “Afetlerde Türkiye’nin Ortak Gücü” anlayışıyla afet yönetimi'

Kaynak: AFAD | Tarih: 2023-11-01

Master'da aynı tarih + AFAD: 1 aday
Master'da ±3 gün AFAD: 1 aday

  doc_id: afad_0032 | tarih: 2023-11-01
  text: 'Türkiye’nin afet yönetimi ve uluslararası yardım kurumu: AFAD Afet ve acil durumlara ilişkin ülkenin yetkin ve yetkili teşkilatıdır.\n\nAFAD, “Afetlerde'

 Toplam corpus: 2139
 Annotation eşleşme: 300/300
